<a href="https://colab.research.google.com/github/romeurf/DipRadar/blob/main/ml_training/DipRadar_Backtest_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 DipRadar — Strategy Backtest (Colab)

Backtest da estratégia **DipRadar** usando **OOF scores** gerados por walk-forward CV.

### Lógica
- Gera OOF scores (out-of-fold) com o **mesmo modelo champion do treino** — **sem lookahead bias**
- Entra numa posição no dia do alerta (preço de fecho)
- Sai ao fim de **60 dias de calendário** (horizon do modelo)
- Filtra apenas alertas com `oof_score >= THRESHOLD`
- Compara contra **Buy & Hold SPY** no mesmo período

### Fluxo
1. Célula 1 — Setup
2. Célula 2 — Carregar `ml_training_base.parquet`
3. Célula 3 — **Gerar OOF scores** via walk-forward (LightGBM, alvo = `alpha_60d`)
4. Célula 4 — Parâmetros do backtest
5. Célula 5 — Simulação de trades
6. Célula 6 — Métricas (Sharpe, Win Rate, Max DD, Alpha)
7. Célula 7 — Gráficos (equity curve, distribuição, alpha, threshold sweep)
8. Célula 8 — Breakdown por sector e ano
9. Célula 9 — Sensitivity analysis

**Pré-requisito:** `ml_training_base.parquet` gerado pelo notebook de treino.

## ⚙️ Célula 1 — Setup

In [ ]:
import os, subprocess, sys
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_API_KEY')
REPO_OWNER   = 'romeurf'
REPO_NAME    = 'DipRadar'
BRANCH       = 'main'

assert GITHUB_TOKEN, 'Falta GITHUB_TOKEN nos secrets do Colab'

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth=1', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'✅ Repo em {REPO_DIR}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'joblib', 'scikit-learn', 'lightgbm', 'xgboost',
                'pandas', 'pyarrow', 'matplotlib', 'scipy'], check=True)
print('✅ Dependências instaladas')

## 📂 Célula 2 — Carregar parquet

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

REPO_PATH    = Path(REPO_DIR)
PARQUET_PATH = REPO_PATH / 'ml_training_base.parquet'

assert PARQUET_PATH.exists(), (
    f'❌ {PARQUET_PATH} não encontrado.\n'
    'Corre primeiro o notebook de treino.'
)

df = pd.read_parquet(PARQUET_PATH)
df['alert_date'] = pd.to_datetime(df['alert_date'])
df = df.sort_values('alert_date').reset_index(drop=True)

print(f'✅ Dataset: {df.shape}')
print(f'   Colunas: {list(df.columns)}')
print(f'   Período: {df.alert_date.min().date()} → {df.alert_date.max().date()}')

# Verificar colunas necessárias
needed = ['close_60d', 'spy_close_60d', 'alpha_60d', 'alert_date', 'ticker']
missing = [c for c in needed if c not in df.columns]
assert not missing, f'Colunas em falta: {missing}'
print(f'✅ Colunas obrigatórias presentes')

## 🔄 Célula 3 — Gerar OOF scores (walk-forward, sem lookahead)

In [ ]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
# Modelo usado para gerar OOF scores (igual ao champion do treino)
OOF_MODEL     = 'LGBM'   # 'LGBM' | 'XGB' | 'RF'
TARGET_COL    = 'alpha_60d'   # alvo de regressão
N_FOLDS       = 5             # folds expanding-window
PURGE_DAYS    = 90            # purga para evitar leakage (> 60d horizon)
HALF_LIFE     = 365           # decay temporal dos pesos de treino

# Features disponíveis no parquet (todas exceto targets e meta-colunas)
EXCLUDE_COLS = {
    'ticker', 'alert_date', 'sector',
    'close_60d', 'spy_close_60d', 'alpha_60d',
    'max_return_60d', 'max_drawdown_60d',
}
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]
print(f'Features: {len(FEATURE_COLS)}  →  {FEATURE_COLS}')

# ─── WALK-FORWARD OOF ─────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

def make_model(name):
    if name == 'LGBM':
        from lightgbm import LGBMRegressor
        return LGBMRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                             min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                             reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=-1)
    elif name == 'XGB':
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8, verbosity=0, random_state=42)
    else:
        from sklearn.ensemble import RandomForestRegressor
        return RandomForestRegressor(n_estimators=200, max_depth=8, n_jobs=-1, random_state=42)

# Remover rows sem target
df_oof = df.dropna(subset=[TARGET_COL] + FEATURE_COLS[:3]).copy()
df_oof = df_oof.reset_index(drop=True)
print(f'Rows válidas: {len(df_oof):,}')

dates    = df_oof['alert_date']
min_date = dates.min()
max_date = dates.max()
total_days = (max_date - min_date).days

oof_scores = np.full(len(df_oof), np.nan)
fold_stats = []

for k in range(N_FOLDS):
    train_frac = 0.50 + 0.50 * k / N_FOLDS
    test_frac  = 0.50 + 0.50 * (k + 1) / N_FOLDS
    train_end  = min_date + pd.Timedelta(days=int(total_days * train_frac))
    purge_end  = train_end + pd.Timedelta(days=PURGE_DAYS)
    test_end   = min_date + pd.Timedelta(days=int(total_days * test_frac))

    if test_end <= purge_end:
        continue

    train_mask = dates <= train_end
    test_mask  = (dates > purge_end) & (dates <= test_end)

    df_tr = df_oof[train_mask]
    df_te = df_oof[test_mask]

    if len(df_tr) < 50 or len(df_te) < 10:
        print(f'  Fold {k+1}: skip (train={len(df_tr)}, test={len(df_te)})')
        continue

    # Imputar NaN com mediana do treino
    X_tr = df_tr[FEATURE_COLS].values.astype(float)
    X_te = df_te[FEATURE_COLS].values.astype(float)
    medians = np.nanmedian(X_tr, axis=0)
    for i in range(X_tr.shape[1]):
        X_tr[~np.isfinite(X_tr[:, i]), i] = medians[i]
        X_te[~np.isfinite(X_te[:, i]), i] = medians[i]

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    y_tr = df_tr[TARGET_COL].values.astype(float)

    # Temporal sample weights
    days_arr = (train_end - df_tr['alert_date']).dt.days.values.astype(float)
    w_tr = np.exp(-np.log(2) * days_arr / HALF_LIFE)

    model = make_model(OOF_MODEL)
    try:
        model.fit(X_tr, y_tr, sample_weight=w_tr)
    except TypeError:
        model.fit(X_tr, y_tr)

    preds = model.predict(X_te).astype(float)
    test_idx = df_oof.index[test_mask]
    oof_scores[test_idx] = preds

    coverage = (~np.isnan(preds)).sum()
    fold_stats.append({'fold': k+1, 'train': len(df_tr), 'test': len(df_te),
                        'train_end': train_end.date(), 'test_end': test_end.date()})
    print(f'  Fold {k+1}/{N_FOLDS}: train={len(df_tr):,}  test={len(df_te):,}  '
          f'[{train_end.date()} → {test_end.date()}]  preds={coverage}')

df_oof['oof_score'] = oof_scores

# Normalizar scores para [0,1] via rank percentile (facilita threshold)
valid_mask = ~df_oof['oof_score'].isna()
from scipy.stats import rankdata
ranks = rankdata(df_oof.loc[valid_mask, 'oof_score'], method='average')
df_oof.loc[valid_mask, 'oof_score_pct'] = ranks / ranks.max()

covered = valid_mask.sum()
print(f'\n✅ OOF scores gerados: {covered:,}/{len(df_oof):,} rows ({covered/len(df_oof):.1%})')
print(f'   oof_score range (alpha): [{df_oof["oof_score"].min():.4f}, {df_oof["oof_score"].max():.4f}]')
print(f'   oof_score_pct range: [0, 1]')

# Dataset final para backtest: apenas rows com OOF score + targets válidos
df_bt = df_oof.dropna(subset=['oof_score', 'oof_score_pct', 'close_60d', 'spy_close_60d']).copy()
print(f'   Rows válidas para backtest: {len(df_bt):,}')

## 🎛️ Célula 4 — Parâmetros do Backtest

In [ ]:
# ─── PARÂMETROS ───────────────────────────────────────────────────────────────

# Threshold de percentil de OOF score (0.0 a 1.0)
# 0.70 = entrar apenas nos top-30% alertas por score predito
SCORE_THRESHOLD = 0.70

# Capital inicial (para curva de equity)
INITIAL_CAPITAL = 10_000

# Máximo de posições simultâneas
MAX_POSITIONS = 20

# Horizonte de saída (dias de calendário — deve coincidir com o do modelo)
HORIZON_DAYS = 60

# Custo por trade (round-trip: bid/ask + comissão)
TRANSACTION_COST = 0.001  # 0.1%

# Período de início do backtest (parte mais recente = menos viés de sobrevivência)
START_DATE = '2015-01-01'

# Usar oof_score_pct (percentil 0-1) ou oof_score (alpha bruto em $)
# Recomendado: USE_PCT = True  →  threshold em percentil é mais estável
USE_PCT = True
SCORE_COL = 'oof_score_pct' if USE_PCT else 'oof_score'

print(f'✅ Parâmetros:')
print(f'   Score col       : {SCORE_COL}')
print(f'   Threshold       : {SCORE_THRESHOLD} ({"percentil" if USE_PCT else "alpha bruto"})')
print(f'   Capital inicial : ${INITIAL_CAPITAL:,}')
print(f'   Max posições    : {MAX_POSITIONS}')
print(f'   Horizonte saída : {HORIZON_DAYS}d')
print(f'   Período backtest: {START_DATE} →')

## 🔁 Célula 5 — Simulação de Trades

In [ ]:
df_filt = df_bt[
    (df_bt['alert_date'] >= START_DATE) &
    (df_bt[SCORE_COL] >= SCORE_THRESHOLD)
].copy().sort_values('alert_date').reset_index(drop=True)

print(f'Alertas após filtro: {len(df_filt):,}  '
      f'({len(df_filt)/len(df_bt[df_bt["alert_date"]>=START_DATE]):.1%} do universo)')

trades = []
open_positions = {}   # ticker → exit_date

for _, row in df_filt.iterrows():
    entry_date = row['alert_date']
    ticker     = row['ticker']

    # Fechar posições vencidas
    open_positions = {t: d for t, d in open_positions.items() if d > entry_date}

    if ticker in open_positions:         # não duplicar o mesmo ticker
        continue
    if len(open_positions) >= MAX_POSITIONS:
        continue

    exit_date = entry_date + pd.Timedelta(days=HORIZON_DAYS)
    open_positions[ticker] = exit_date

    ret     = float(row['close_60d'])     - TRANSACTION_COST
    spy_ret = float(row['spy_close_60d'])
    alpha   = ret - spy_ret

    trades.append({
        'entry_date'  : entry_date,
        'exit_date'   : exit_date,
        'ticker'      : ticker,
        'oof_score'   : float(row['oof_score']),
        'oof_score_pct': float(row.get('oof_score_pct', np.nan)),
        'ret'         : ret,
        'spy_ret'     : spy_ret,
        'alpha'       : alpha,
        'win'         : ret > 0,
        'beat_spy'    : alpha > 0,
    })

trades_df = pd.DataFrame(trades)
print(f'\n✅ Trades simulados: {len(trades_df):,}')
if len(trades_df):
    print(f'   Período: {trades_df.entry_date.min().date()} → {trades_df.entry_date.max().date()}')

## 📊 Célula 6 — Métricas

In [ ]:
if len(trades_df) == 0:
    print('❌ Nenhum trade. Baixa o SCORE_THRESHOLD.')
else:
    n          = len(trades_df)
    win_rate   = trades_df['win'].mean()
    beat_rate  = trades_df['beat_spy'].mean()
    mean_ret   = trades_df['ret'].mean()
    mean_alpha = trades_df['alpha'].mean()
    std_ret    = trades_df['ret'].std()

    periods_per_year = 252 / HORIZON_DAYS
    sharpe = (mean_ret / std_ret) * np.sqrt(periods_per_year) if std_ret > 0 else 0

    trades_sorted = trades_df.sort_values('entry_date')
    equity     = [INITIAL_CAPITAL]
    spy_equity = [INITIAL_CAPITAL]
    for _, t in trades_sorted.iterrows():
        equity.append(equity[-1] * (1 + t['ret']))
        spy_equity.append(spy_equity[-1] * (1 + t['spy_ret']))

    eq_arr = np.array(equity)
    peak   = np.maximum.accumulate(eq_arr)
    max_dd = ((eq_arr - peak) / peak).min()

    total_ret     = equity[-1]     / INITIAL_CAPITAL - 1
    spy_total_ret = spy_equity[-1] / INITIAL_CAPITAL - 1

    print('=' * 58)
    print(f'  DipRadar Backtest  |  {SCORE_COL} >= {SCORE_THRESHOLD}')
    print('=' * 58)
    print(f'  Trades              : {n:,}')
    print(f'  Win Rate            : {win_rate:.1%}')
    print(f'  Beat SPY Rate       : {beat_rate:.1%}')
    print(f'  Avg Return / trade  : {mean_ret:+.2%}')
    print(f'  Avg Alpha  / trade  : {mean_alpha:+.2%}')
    print(f'  Sharpe (annualised) : {sharpe:.2f}')
    print(f'  Max Drawdown        : {max_dd:.1%}')
    print(f'  Total Return (strat): {total_ret:+.1%}')
    print(f'  Total Return (SPY)  : {spy_total_ret:+.1%}')
    print('=' * 58)

## 📉 Célula 7 — Gráficos

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 130, 'font.family': 'monospace'})

if len(trades_df) > 0:
    dates = [trades_sorted['entry_date'].iloc[0]] + list(trades_sorted['exit_date'])

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(
        f'DipRadar Backtest  |  {SCORE_COL}>={SCORE_THRESHOLD}  |  {n} trades  |  Sharpe={sharpe:.2f}',
        fontsize=12, fontweight='bold'
    )

    # 1. Equity curve
    ax1 = axes[0, 0]
    ax1.plot(dates, equity,     label='DipRadar', color='#2196F3', linewidth=2)
    ax1.plot(dates, spy_equity, label='SPY B&H',  color='#FF5722', linewidth=1.5, linestyle='--')
    ax1.set_title('Curva de Equity (equal-weight, reinvestimento)')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax1.legend(); ax1.grid(alpha=0.3)

    # 2. Distribuição retornos
    ax2 = axes[0, 1]
    ax2.hist(trades_df['ret']     * 100, bins=40, alpha=0.6, label='DipRadar', color='#2196F3', edgecolor='white')
    ax2.hist(trades_df['spy_ret'] * 100, bins=40, alpha=0.5, label='SPY same period', color='#FF5722', edgecolor='white')
    ax2.axvline(0, color='black', linewidth=1, linestyle=':')
    ax2.axvline(mean_ret * 100, color='#2196F3', linewidth=1.5, linestyle='--', label=f'Avg {mean_ret*100:+.1f}%')
    ax2.set_xlabel('Retorno 60d (%)'); ax2.set_title('Distribuição de Retornos por Trade')
    ax2.legend(); ax2.grid(alpha=0.3)

    # 3. Alpha por trade (sorted)
    ax3 = axes[1, 0]
    sorted_alpha = trades_df['alpha'].sort_values().values * 100
    colors = ['#4CAF50' if a > 0 else '#F44336' for a in sorted_alpha]
    ax3.bar(range(len(sorted_alpha)), sorted_alpha, color=colors, width=1.0, alpha=0.8)
    ax3.axhline(0, color='black', linewidth=0.8)
    ax3.axhline(mean_alpha * 100, color='orange', linewidth=1.5, linestyle='--', label=f'Avg {mean_alpha*100:+.2f}%')
    ax3.set_xlabel('Trade (ordenado por alpha)'); ax3.set_ylabel('Alpha vs SPY (%)')
    ax3.set_title(f'Alpha por Trade  |  Beat SPY: {beat_rate:.1%}')
    ax3.legend(); ax3.grid(alpha=0.3)

    # 4. Win Rate vs threshold
    ax4 = axes[1, 1]
    df_sweep_chart = df_bt[df_bt['alert_date'] >= START_DATE].copy()
    thresholds = np.linspace(0.50, 0.95, 20)
    wr_pts = []
    for thr in thresholds:
        sub = df_sweep_chart[df_sweep_chart[SCORE_COL] >= thr]
        if len(sub) < 10: continue
        wr_pts.append((thr, (sub['close_60d'] > 0).mean() * 100, len(sub)))
    if wr_pts:
        tv, wv, nv = zip(*wr_pts)
        ax4.plot(tv, wv, color='#2196F3', linewidth=2, label='Win Rate %')
        ax4.axhline(50, color='black', linestyle=':', linewidth=1)
        ax4.axvline(SCORE_THRESHOLD, color='orange', linestyle='--', linewidth=1.5,
                    label=f'Threshold={SCORE_THRESHOLD}')
        ax4.set_xlabel(SCORE_COL); ax4.set_ylabel('Win Rate (%)')
        ax4.set_title('Win Rate vs Score Threshold')
        ax4b = ax4.twinx()
        ax4b.bar(tv, nv, width=0.02, alpha=0.2, color='grey')
        ax4b.set_ylabel('# Alertas')
        ax4.legend(loc='upper left'); ax4.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/backtest_results.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('✅ Gráfico → /content/backtest_results.png')

## 🔍 Célula 8 — Breakdown por Sector e Ano

In [ ]:
if len(trades_df) > 0:
    tf = trades_df.copy()

    if 'sector' in df_bt.columns:
        sec_map = df_bt[['ticker','sector']].drop_duplicates().set_index('ticker')['sector']
        tf['sector'] = tf['ticker'].map(sec_map).fillna('Unknown')
    else:
        tf['sector'] = 'Unknown'

    tf['year'] = tf['entry_date'].dt.year

    def fmt_stats(grp):
        s = grp.agg(trades=('ret','count'), win_rate=('win','mean'),
                    avg_ret=('ret','mean'), avg_alpha=('alpha','mean'))
        s['win_rate'] = s['win_rate'].map('{:.1%}'.format)
        s['avg_ret']  = s['avg_ret'].map('{:+.2%}'.format)
        s['avg_alpha']= s['avg_alpha'].map('{:+.2%}'.format)
        return s

    print('\n── Por Sector (ordenado por avg_alpha) ────────────────────────────')
    print(fmt_stats(tf.groupby('sector')).sort_values('avg_alpha',ascending=False).to_string())

    print('\n── Por Ano ─────────────────────────────────────────────────────────')
    print(fmt_stats(tf.groupby('year')).to_string())

    tf.to_csv('/content/backtest_trades.csv', index=False)
    print('\n✅ Trades CSV → /content/backtest_trades.csv')

## 🧪 Célula 9 — Sensitivity: threshold sweep

In [ ]:
df_sw = df_bt[df_bt['alert_date'] >= START_DATE].copy()
sweep_thresholds = np.arange(0.50, 0.96, 0.05)
results = []

for thr in sweep_thresholds:
    sub = df_sw[df_sw[SCORE_COL] >= thr]
    if len(sub) < 10:
        break
    ar  = sub['close_60d'].mean() - TRANSACTION_COST
    aa  = (sub['close_60d'] - sub['spy_close_60d']).mean()
    std = sub['close_60d'].std()
    sh  = (ar / std) * np.sqrt(252 / HORIZON_DAYS) if std > 0 else 0
    results.append({
        'threshold'  : f'{thr:.2f}',
        'n_alertas'  : len(sub),
        'win_rate'   : f'{(sub["close_60d"]>0).mean():.1%}',
        'beat_spy'   : f'{(sub["alpha_60d"]>0).mean():.1%}',
        'avg_ret'    : f'{ar:+.2%}',
        'avg_alpha'  : f'{aa:+.2%}',
        'sharpe'     : f'{sh:.2f}',
    })

sweep_df = pd.DataFrame(results)
print('\n── Sensitivity Sweep ──────────────────────────────────────────────────')
print(sweep_df.to_string(index=False))
print('\n💡 Escolhe o threshold com melhor tradeoff  n_alertas × sharpe × avg_alpha')